### Exercício 1

In [ ]:
# Instalando as bibliotecas necessárias para a execução otimizada da simulação
!pip install numba

In [6]:
import math
import random
import time
from numba import jit

# O @jit avisa pro Numba compilar essa função para rodar na velocidade da luz
@jit(nopython=True)
def simular_fila_mm1(n, lambd, mu):
    soma_espera = 0.0
    soma_espera_quadrado = 0.0
    
    t_chegada_atual = 0.0
    t_saida_anterior = 0.0
    
    beta_c = 1.0 / lambd
    beta_s = 1.0 / mu
    
    for _ in range(n):
        # 1. Chegada (T_c)
        u_c = random.random()
        tc = -beta_c * math.log(1.0 - u_c)
        t_chegada_atual += tc
        
        # 2. Serviço (T_s)
        u_s = random.random()
        ts = -beta_s * math.log(1.0 - u_s)
        
        # 3. Espera
        espera = max(0.0, t_saida_anterior - t_chegada_atual)
        soma_espera += espera
        soma_espera_quadrado += espera ** 2
        
        # 4. Atualizar relógio
        t_saida_anterior = t_chegada_atual + espera + ts

    # Cálculo final
    media_espera = soma_espera / n
    variancia = (soma_espera_quadrado / n) - (media_espera ** 2)
    desvio_padrao = math.sqrt(max(0.0, variancia))
    h = 1.96 * (desvio_padrao / math.sqrt(n))
    
    return media_espera, h

# Parâmetros pedidos pelo professor
lambd = 9.0
mu = 10.0
tamanhos_n = [10**3, 10**5, 10**7, 10**9]

print(f"Simulação Fila M/M/1 (lambda={lambd}, mu={mu})")

for n in tamanhos_n:
    inicio = time.time()
    
    media, h = simular_fila_mm1(n, lambd, mu)
    
    fim = time.time()
    tempo_execucao = fim - inicio
    
    print(f"n = {n:<10} | Espera: {media:.4f} s | IC 95%: [{media - h:.4f}, {media + h:.4f}] | H = {h:.6f} | Levou: {tempo_execucao:.2f}s")

Simulação Fila M/M/1 (lambda=9.0, mu=10.0)
n = 1000       | Espera: 0.4897 s | IC 95%: [0.4575, 0.5220] | H = 0.032230 | Levou: 0.09s
n = 100000     | Espera: 0.8877 s | IC 95%: [0.8818, 0.8935] | H = 0.005842 | Levou: 0.00s
n = 10000000   | Espera: 0.8981 s | IC 95%: [0.8975, 0.8988] | H = 0.000614 | Levou: 0.28s
n = 1000000000 | Espera: 0.9001 s | IC 95%: [0.9001, 0.9002] | H = 0.000062 | Levou: 28.66s


### Exercício 2

In [7]:
@jit(nopython=True)
def simular_chow_robbins(d, lambd, mu):
    t_chegada_atual = 0.0
    t_saida_anterior = 0.0
    
    beta_c = 1.0 / lambd
    beta_s = 1.0 / mu
    
    n = 0
    media_espera = 0.0
    M2 = 0.0
    
    # Loop infinito que só para quando a condição H <= d for atingida
    while True:
        n += 1
        
        # 1. Chegada
        u_c = random.random()
        tc = -beta_c * math.log(1.0 - u_c)
        t_chegada_atual += tc
        
        # 2. Serviço
        u_s = random.random()
        ts = -beta_s * math.log(1.0 - u_s)
        
        # 3. Espera
        espera = max(0.0, t_saida_anterior - t_chegada_atual)
        
        # 4. Atualizar relógio
        t_saida_anterior = t_chegada_atual + espera + ts

        # Algoritmo de Welford para Média e Variância em tempo real
        delta = espera - media_espera
        media_espera += delta / n
        delta2 = espera - media_espera
        M2 += delta * delta2
        
        # Só começamos a testar a parada depois de 1000 clientes 
        if n > 1000:
            variancia = M2 / (n - 1)
            desvio_padrao = math.sqrt(max(0.0, variancia))
            h = 1.96 * (desvio_padrao / math.sqrt(n))
            
            if h <= d:
                return n, media_espera, h

lambd = 9.0
mu = 10.0
valores_d = [1.0, 0.5, 0.1, 0.05]

print("Simulação Fila M/M/1: Regra de Chow e Robbins")

for d in valores_d:
    inicio = time.time()
    
    n_final, media, h_final = simular_chow_robbins(d, lambd, mu)
    
    fim = time.time()
    tempo = fim - inicio
    
    print(f"Alvo (d) = {d:<4} | Clientes (n) necessários: {n_final:<8} | Espera: {media:.4f}s | H final: {h_final:.5f} | Levou: {tempo:.2f}s")

Simulação Fila M/M/1: Regra de Chow e Robbins
Alvo (d) = 1.0  | Clientes (n) necessários: 1001     | Espera: 0.3661s | H final: 0.02366 | Levou: 0.09s
Alvo (d) = 0.5  | Clientes (n) necessários: 1001     | Espera: 0.6631s | H final: 0.04082 | Levou: 0.00s
Alvo (d) = 0.1  | Clientes (n) necessários: 1001     | Espera: 0.8273s | H final: 0.05309 | Levou: 0.00s
Alvo (d) = 0.05 | Clientes (n) necessários: 1001     | Espera: 0.6631s | H final: 0.04248 | Levou: 0.00s


### Exercício 3

In [8]:
@jit(nopython=True)
def simular_tamanho_relativo(gamma, lambd, mu):
    t_chegada_atual = 0.0
    t_saida_anterior = 0.0
    
    beta_c = 1.0 / lambd
    beta_s = 1.0 / mu
    
    n = 0
    media_espera = 0.0
    M2 = 0.0
    
    while True:
        n += 1
        
        # 1. Chegada
        u_c = random.random()
        tc = -beta_c * math.log(1.0 - u_c)
        t_chegada_atual += tc
        
        # 2. Serviço
        u_s = random.random()
        ts = -beta_s * math.log(1.0 - u_s)
        
        # 3. Espera
        espera = max(0.0, t_saida_anterior - t_chegada_atual)
        
        # 4. Atualizar relógio
        t_saida_anterior = t_chegada_atual + espera + ts

        # Algoritmo de Welford (Média e Variância)
        delta = espera - media_espera
        media_espera += delta / n
        delta2 = espera - media_espera
        M2 += delta * delta2
        
        # Testar a parada (apenas após 1000 clientes e garantindo que a média não seja zero)
        if n > 1000 and media_espera > 0:
            variancia = M2 / (n - 1)
            desvio_padrao = math.sqrt(max(0.0, variancia))
            h = 1.96 * (desvio_padrao / math.sqrt(n))
            
            # Regra de Parada do Tamanho Relativo
            if (h / media_espera) <= gamma:
                return n, media_espera, h

lambd = 9.0
mu = 10.0
gamma = 0.05

print("Simulação Fila M/M/1: Tamanho Relativo do IC")

inicio = time.time()

n_final, media, h_final = simular_tamanho_relativo(gamma, lambd, mu)

fim = time.time()
tempo = fim - inicio

print(f"Alvo (gamma) = {gamma*100}% | Clientes (n): {n_final} | Espera Média: {media:.4f}s")
print(f"H final: {h_final:.5f} | Razão H/Media: {(h_final/media):.5f} | Levou: {tempo:.2f}s")

Simulação Fila M/M/1: Tamanho Relativo do IC
Alvo (gamma) = 5.0% | Clientes (n): 1239 | Espera Média: 1.3809s
H final: 0.06901 | Razão H/Media: 0.04997 | Levou: 0.09s
